In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
import timm
from tqdm import tqdm
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

from nodule_dataset import NoduleDatasetYOLO, collate_fn


c:\Users\Edrill-LT\Documents\Projects\Python\Thoracic-Disease-Classifier-ResNet50\.torch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==================== Configuration ====================
CSV_PATH = './dataset_nodule21/cxr_images/proccessed_data/metadata.csv'
IMG_DIR = './dataset_nodule21/cxr_images/proccessed_data/images'
NUM_CLASSES = 2  # background + nodule
BATCH_SIZE = 4
NUM_EPOCHS = 25
LR = 0.001
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 4
TRAIN_SPLIT = 0.7  # 70% train
VAL_SPLIT = 0.15   # 15% val
TEST_SPLIT = 0.15  # 15% test

print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
# ==================== Focal Loss for Class Imbalance ====================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        
    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()


In [ ]:
# ==================== Create YOLO Model ====================
print("\nInitializing YOLOv8-small model...")

# Load pretrained YOLOv8-small model
model = YOLO('yolov8s.pt')

In [ ]:
# ==================== Dataset Split by Unique Images ====================
print("Loading dataset and splitting by unique images...")

# Read CSV to get unique image names
df = pd.read_csv(CSV_PATH, index_col=0)
unique_images = df['img_name'].unique()

print(f"Total unique images: {len(unique_images)}")
print(f"Total annotations (including multiple nodules per image): {len(df)}")

# Split unique image names: 70% train, 15% val, 15% test
train_images, temp_images = train_test_split(
    unique_images,
    train_size=TRAIN_SPLIT,
    random_state=42,
    shuffle=True
)

# Split remaining 30% into val (50% of 30% = 15%) and test (50% of 30% = 15%)
val_images, test_images = train_test_split(
    temp_images,
    train_size=0.5,
    random_state=42,
    shuffle=True
)

print(f"\nTrain images: {len(train_images)} ({len(train_images)/len(unique_images)*100:.1f}%)")
print(f"Val images: {len(val_images)} ({len(val_images)/len(unique_images)*100:.1f}%)")
print(f"Test images: {len(test_images)} ({len(test_images)/len(unique_images)*100:.1f}%)")

# Create datasets with specific image subsets
train_dataset = NoduleDataset(
    CSV_PATH, 
    IMG_DIR, 
    img_names=train_images,
    transforms=get_train_transforms()
)

val_dataset = NoduleDataset(
    CSV_PATH, 
    IMG_DIR, 
    img_names=val_images,
    transforms=get_val_transforms()
)

test_dataset = NoduleDataset(
    CSV_PATH, 
    IMG_DIR, 
    img_names=test_images,
    transforms=get_val_transforms()
)

# Verify no overlap
train_set = set(train_dataset.img_names)
val_set = set(val_dataset.img_names)
test_set = set(test_dataset.img_names)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True
)


Loading dataset and splitting by unique images...
Total unique images: 4882
Total annotations (including multiple nodules per image): 5224

Train images: 3417 (70.0%)
Val images: 732 (15.0%)
Test images: 733 (15.0%)
✓ No data leakage: train, val, and test sets have unique images


c:\Users\Edrill-LT\Documents\Projects\Python\Thoracic-Disease-Classifier-ResNet50\.torch\lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
c:\Users\Edrill-LT\Documents\Projects\Python\Thoracic-Disease-Classifier-ResNet50\.torch\lib\site-packages\albumentations\core\composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()
c:\Users\Edrill-LT\Documents\Projects\Python\Thoracic-Disease-Classifier-ResNet50\.torch\lib\site-packages\albumentations\core\composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()


In [6]:
# ==================== Model, Optimizer, Scheduler ====================
model = create_model(NUM_CLASSES)
model.to(DEVICE)

# Optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=LR, momentum=0.9, weight_decay=0.0005)

# Learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.1)

In [ ]:
# ==================== Training Loop ====================
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*50}")
    
    # ========== Training ==========
    model.train()
    train_loss = 0.0
    
    pbar = tqdm(train_loader, desc='Training')
    for images, targets in pbar:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        
        # Forward pass
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        # Backward pass
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        train_loss += losses.item()
        pbar.set_postfix({'loss': losses.item()})
    
    avg_train_loss = train_loss / len(train_loader)
    
    # ========== Validation ==========
    model.train()  # Keep in train mode for loss calculation
    val_loss = 0.0
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc='Validation')
        for images, targets in pbar:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            val_loss += losses.item()
            pbar.set_postfix({'loss': losses.item()})
    
    avg_val_loss = val_loss / len(val_loader)
    
    # Update learning rate
    lr_scheduler.step()
    
    # Print epoch summary
    print(f"\nTrain Loss: {avg_train_loss:.4f}")
    print(f"Val Loss: {avg_val_loss:.4f}")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),``
            'val_loss': avg_val_loss,
            'train_images': train_images,
            'val_images': val_images
        }, 'best_model.pth')
        print(f"✓ Saved best model (val_loss: {avg_val_loss:.4f})")

print("\n" + "="*50)
print("Training completed!")
print(f"Best validation loss: {best_val_loss:.4f}")
print("="*50)


Epoch 1/25


Validation:   0%|          | 0/183 [00:00<?, ?it/s]

In [ ]:
# ==================== Inference Example ====================
print("\nTesting inference...")
model.eval()
checkpoint = torch.load('best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])

with torch.no_grad():
    test_img, test_target = val_dataset[0]
    prediction = model([test_img.to(DEVICE)])
    
    print(f"\nPredicted boxes: {len(prediction[0]['boxes'])}")
    print(f"Predicted scores: {prediction[0]['scores'][:5].cpu().numpy()}")
    print(f"Ground truth boxes: {len(test_target['boxes'])}")

In [ ]:
# ==================== Test Set Evaluation ====================
print("\n" + "="*50)
print("Evaluating on Test Set")
print("="*50)

model.eval()
test_predictions = []
test_targets = []

with torch.no_grad():
    pbar = tqdm(test_loader, desc='Testing')
    for images, targets in pbar:
        images = [img.to(DEVICE) for img in images]
        predictions = model(images)
        
        test_predictions.extend(predictions)
        test_targets.extend(targets)

print(f"\nTest set evaluation completed on {len(test_dataset)} images")
print("Predictions saved for further analysis")